# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# encoding='latin-1' because the file contains non-UTF-8 characters
df = pd.read_csv('AviationData.csv', encoding='latin-1', low_memory=False)
print(df.shape)
df.info()
print(df.isna().mean().sort_values(ascending=False).round(3).head(15))
df.describe()

(88889, 31)
<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-nul

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
# Cleaning assumptions (filters), in order:
# 1) Keep real "Accident" events only (incidents are minor events, not what the insurer tracks)
# 2) Keep events from 1983 onwards (40-year maximum lifetime of a make/model)
# 3) Keep professional builds only (Amateur.Built == 'No')
# 4) Keep airplanes only (no helicopters, gliders, balloons...)
df = df[df['Investigation.Type'] == 'Accident'].copy()
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df = df[df['Event.Date'] >= '1983-01-01']
df = df[df['Amateur.Built'] == 'No']
print('after date/accident/professional filters:', df.shape)

# Aircraft.Category is missing for most rows -> impute in two steps
df['Aircraft.Category'] = df['Aircraft.Category'].replace('Unknown', np.nan)
# a) a Make+Model seen elsewhere with a known category gets that category (most frequent one)
key = df['Make'].str.upper().str.strip() + ' | ' + df['Model'].str.upper().str.strip()
lookup = df['Aircraft.Category'].groupby(key).agg(lambda s: s.mode().iloc[0] if s.notna().any() else np.nan)
df['Aircraft.Category'] = df['Aircraft.Category'].fillna(key.map(lookup))
# b) still missing: turbine engines other than turbo shaft (helicopters) are airplanes
turbine = df['Engine.Type'].isin(['Turbo Prop', 'Turbo Fan', 'Turbo Jet', 'Geared Turbofan'])
df.loc[df['Aircraft.Category'].isna() & turbine, 'Aircraft.Category'] = 'Airplane'
# Remaining unknown categories are dropped (we cannot be sure they are airplanes)
df = df[df['Aircraft.Category'] == 'Airplane']
print('airplanes:', df.shape)

after date/accident/professional filters: (73286, 31)


airplanes: (59148, 31)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
# Passengers = fatal + serious + minor + uninjured (everyone on board with a recorded outcome)
# Assumption: if at least one of the four counts is recorded, a missing count means 0;
# if all four are missing (or the total is 0) the fraction cannot be computed -> drop the row
inj_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
df = df.dropna(subset=inj_cols, how='all').copy()
df[inj_cols] = df[inj_cols].fillna(0)
df['Total.Passengers'] = df[inj_cols].sum(axis=1)
df = df[df['Total.Passengers'] > 0].copy()

# Derived metric: fraction of people on board fatally or seriously injured
df['Fatal.Serious.Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']
print(df[['Total.Passengers', 'Fatal.Serious.Fraction']].describe())

       Total.Passengers  Fatal.Serious.Fraction
count      58828.000000            58828.000000
mean           4.982831                0.277381
std           23.672407                0.431766
min            1.000000                0.000000
25%            1.000000                0.000000
50%            2.000000                0.000000
75%            2.000000                0.750000
max          699.000000                1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
# Aircraft.damage: 'Unknown' is not informative -> NaN, then drop (damage is a key outcome)
print(df['Aircraft.damage'].value_counts(dropna=False))
df['Aircraft.damage'] = df['Aircraft.damage'].replace('Unknown', np.nan)
df = df.dropna(subset=['Aircraft.damage']).copy()

# Derived column: 1 if the aircraft was destroyed, else 0
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)
print(df['Destroyed'].mean().round(3), df.shape)

Aircraft.damage
Substantial    45320
Destroyed      12084
NaN              921
Minor            439
Unknown           64
Name: count, dtype: int64
0.209 (57843, 34)


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
# Make cleaning tasks:
# 1) drop NaN, 2) strip spaces and upper-case (CESSNA vs Cessna),
# 3) remove company suffixes (INC, CORP, CO...), 4) merge known spelling variants of the same maker
df = df.dropna(subset=['Make']).copy()
df['Make'] = (df['Make'].str.upper().str.strip()
              .str.replace(r'[.,]', '', regex=True)
              .str.replace(r'\s+(INC|CORP|CORPORATION|CO|COMPANY|LTD)$', '', regex=True))
make_map = {
    'BEECHCRAFT': 'BEECH', 'THE BEECH': 'BEECH', 'RAYTHEON AIRCRAFT COMPANY': 'BEECH',
    'DEHAVILLAND': 'DE HAVILLAND', 'CIRRUS DESIGN': 'CIRRUS', 'GRUMMAN AMERICAN': 'GRUMMAN',
    'AIRBUS INDUSTRIE': 'AIRBUS', 'PIPER AIRCRAFT': 'PIPER', 'CESSNA AIRCRAFT': 'CESSNA',
    'THE BOEING': 'BOEING', 'BOEING OF CANADA': 'BOEING', 'MOONEY AIRCRAFT': 'MOONEY',
    'AVIAT AIRCRAFT': 'AVIAT', 'AMERICAN CHAMPION AIRCRAFT': 'CHAMPION', 'GRUMMAN AMERICAN AVN': 'GRUMMAN',
    'GRUMMAN ACFT ENG COR-SCHWEIZER': 'SCHWEIZER', 'GRUMMAN-SCHWEIZER': 'SCHWEIZER',
    'ROCKWELL INTERNATIONAL': 'ROCKWELL', 'DIAMOND AIRCRAFT IND': 'DIAMOND', 'DIAMOND AIRCRAFT': 'DIAMOND',
    'ERCOUPE (ENG & RESEARCH CORP)': 'ERCOUPE', 'GATES LEARJET': 'LEARJET', 'RAYTHEON AIRCRAFT': 'BEECH',
}
df['Make'] = df['Make'].replace(make_map)

# Keep makes with enough events (threshold 50)
make_counts = df['Make'].value_counts()
df = df[df['Make'].isin(make_counts[make_counts >= 50].index)].copy()
print(df['Make'].nunique(), 'makes kept;', df.shape)
print(df['Make'].value_counts().head(15))

45 makes kept; (54660, 34)
Make
CESSNA          24489
PIPER           13428
BEECH            4450
MOONEY           1192
GRUMMAN          1168
BELLANCA          909
AIR TRACTOR       863
BOEING            779
AERONCA           527
CHAMPION          526
MAULE             488
CIRRUS            426
STINSON           390
DE HAVILLAND      376
LUSCOMBE          362
Name: count, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
# Model: drop NaN and standardize the text
df = df.dropna(subset=['Model']).copy()
df['Model'] = df['Model'].str.upper().str.strip()

# Are model labels unique to each make? Check labels shared by several makes
shared = df.groupby('Model')['Make'].nunique()
print('Model labels used by more than one make:', (shared > 1).sum())

# They are not unique (e.g. '172' / '100' exist for several makes) -> unique plane type identifier
df['Make.Model'] = df['Make'] + ' ' + df['Model']
print(df['Make.Model'].value_counts().head(10))

Model labels used by more than one make: 105
Make.Model
CESSNA 152         2200
CESSNA 172         1611
CESSNA 172N        1079
PIPER PA-28-140     856
CESSNA 172M         751
CESSNA 150          739
CESSNA 172P         659
CESSNA 182          602
CESSNA 180          592
CESSNA 150M         548
Name: count, dtype: int64


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
# Standardize the "other factors" columns: strings upper/lower consistent, 'unknown' codes -> NaN
df['Weather.Condition'] = df['Weather.Condition'].str.upper().str.strip().replace({'UNK': np.nan, 'UNKNOWN': np.nan})
df['Engine.Type'] = df['Engine.Type'].str.strip().replace({'Unknown': np.nan, 'NONE': np.nan, 'LR': np.nan, 'UNK': np.nan})
# An airplane cannot have 0 engines -> treat 0 as a missing value
df['Number.of.Engines'] = df['Number.of.Engines'].replace(0, np.nan)
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip().replace({'Unknown': np.nan})
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip().replace({'Unknown': np.nan})

for c in ['Weather.Condition', 'Engine.Type', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(df[c].value_counts(dropna=False), '\n')

Weather.Condition
VMC    48217
IMC     4351
NaN     2081
Name: count, dtype: int64 

Engine.Type
Reciprocating    48203
NaN               3003
Turbo Prop        2400
Turbo Fan          794
Turbo Jet          237
Turbo Shaft         12
Name: count, dtype: int64 

Number.of.Engines
1.0    45720
2.0     6669
NaN     2000
4.0      136
3.0      124
Name: count, dtype: int64 

Purpose.of.flight
Personal                     32679
Instructional                 7681
NaN                           5155
Aerial Application            3170
Business                      2641
Positioning                    901
Other Work Use                 483
Ferry                          478
Aerial Observation             365
Executive/corporate            299
Public Aircraft                258
Skydiving                      167
Flight Test                    112
Banner Tow                      96
Public Aircraft - Federal       43
Glider Tow                      33
Public Aircraft - State         26
Firefighting 

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
# Drop columns with more than 50% missing values (e.g. coordinates, airport, air carrier, schedule)
na_frac = df.isna().mean().sort_values(ascending=False)
print(na_frac.head(12).round(2))
df = df.drop(columns=na_frac[na_frac > 0.5].index)
print(df.shape)

Schedule                 0.89
Air.carrier              0.83
FAR.Description          0.69
Longitude                0.61
Latitude                 0.61
Airport.Code             0.39
Airport.Name             0.36
Broad.phase.of.flight    0.28
Publication.Date         0.17
Purpose.of.flight        0.09
Engine.Type              0.05
Report.Status            0.05
dtype: float64
(54649, 30)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [10]:
df.to_csv('AviationData_cleaned.csv', index=False)
print('saved', df.shape)

saved (54649, 30)
